# Dynamic Partition Pruning (DPP)

When a *large, partitioned* fact table is joined to a *small* dimension table that carries a filter, Spark can defer scanning the fact table partitions until it has evaluated the filter on the dimension side. At runtime it then reads only the matching partitions of the fact table — the rest are skipped entirely. This is called **Dynamic Partition Pruning** (DPP), and it can turn a full-table scan into a tiny one without any rewrite of the query.

Three things have to be true for DPP to fire:
* the fact table is partitioned on a column that participates in the join condition
* there is a filter on the dimension side that meaningfully reduces the join key set
* the dimension side is broadcast (`BroadcastHashJoin`) — by default DPP only reuses broadcast results

Tasks:
1. Prepare a country-partitioned fact table from the `users` data.
2. Build a small dimension table that maps countries to regions.
3. Run the join with DPP disabled and read the physical plan.
4. Re-enable DPP and observe the `dynamicpruningexpression` in the scan node.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, trim, split, is_valid_utf8, regexp_replace, lower

import os

In [ ]:
spark = (
    SparkSession
    .builder
    .appName('Dynamic Partition Pruning')
    .getOrCreate()
)

In [ ]:
print(spark.version)

#### Notes about the session:

* we turn AQE off so the physical plan we read with `explain` does not change between the planning and execution phases — DPP is a planning-time optimization and it is much easier to see in a static plan
* the join later forces a broadcast hash join via `.hint('broadcast')` on the dimension side, so DPP can reuse that broadcast

In [ ]:
spark.conf.set('spark.sql.adaptive.enabled', False)

In [ ]:
base_path = os.getcwd()

project_path = ('/').join(base_path.split('/')[0:-3]) 

answers_input_path = os.path.join(project_path, 'data/answers')
users_input_path = os.path.join(project_path, 'data/users')

users_by_location_path = os.path.join(project_path, 'output/user_by_location')
year_dim_path = os.path.join(project_path, 'output/dim_year')
location_dim_path = os.path.join(project_path, 'output/dim_location')

### Task 1: Prepare a country-partitioned fact table

Read the users data, normalize the `location` column (drop nulls and empty strings, keep only entries that are plain ASCII letters, lowercase them, replace any non-alphanumeric character with an underscore), and write the result to disk partitioned by `location`. This is the partitioned fact table we will query in the rest of the notebook.

Hint:
* `repartition('location')` before writing to get one file per location folder
* `write.partitionBy('location')` to lay the data out as `location=france/`, `location=greece/`, ...
* DPP works on filesystem partitioning — the partition column must be visible to the scan

In [ ]:
usersDF = (
    spark.read.parquet(users_input_path)
)

(
    usersDF
    .filter(col('location').isNotNull())
    .filter(trim(col('location')) != '')
    .filter(is_valid_utf8(col('location')))
    .filter(col('location').rlike("^[a-zA-Z ]+$"))
    .withColumn('location', lower('location'))
    .withColumn(
        'location',
        regexp_replace(col('location'), r'[^a-z0-9]', '_')
    )
    .repartition('location')
    .write
    .mode('overwrite')
    .partitionBy('location')
    .option('path', users_by_location_path)
    .save()
)

In [ ]:
users_fact = spark.read.parquet(users_by_location_path)

### Task 2: Build a small dimension table mapping countries to regions

Construct a tiny DataFrame with `country` and `region` columns covering a handful of countries. This stands in for a real geography dimension — small enough to be broadcast — and carries the effective filter on `region` that DPP will translate into a partition-pruning predicate on the fact side.

Hint:
* build the dimension from a literal list of `(country, region)` pairs and write it to disk so the join is reading two real Parquet sources
* in a real warehouse this would be a much larger geography dimension with cities, countries, regions, continents, ...

In [ ]:
location_dim = spark.createDataFrame([
        ('Sweden', 'Europe'), ('Greece', 'Europe'), ('France', 'Europe'), ('Norway', 'Europe'),
        ('USA', 'North America'),
        ('Singapore', 'Asia'), ('China', 'Asia'), ('Israel', 'Asia'), ('Pakistan', 'Asia'), ('Thailand', 'Asia'),
        ('Brazil', 'South America'),
        ('Uganda', 'Africa'), ('Egypt', 'Africa'),
    ], ['country', 'region'])

(
    location_dim
    .write
    .mode('overwrite')
    .option('path', location_dim_path)
    .save()
)

regions_dim = spark.read.parquet(location_dim_path)

In [ ]:
regions_dim.show()

### Task 3: Baseline — run the join with DPP disabled

Disable DPP, rename the fact table's `location` column to `country`, join with the regions dimension filtered to `region = 'Europe'`, look at the physical plan, and trigger execution with the `noop` writer. The dimension carries a `.hint('broadcast')` so the join is a `BroadcastHashJoin` regardless of the threshold — that way the only thing changing between this task and the next is the DPP toggle.

In the plan, the `Scan parquet` node for the user-by-location fact should show `PartitionFilters: []` — meaning every country partition is read, and the dimension's filtering effect is only applied *after* the scan.

Hint:
* toggle: `spark.sql.optimizer.dynamicPartitionPruning.enabled = false`
* call `.explain()` on the joined DataFrame and locate the `Scan parquet` node
* after execution, go to the Stages tab — the input size of the fact-side stage is the **full** dataset

In [ ]:
spark.conf.get('spark.sql.optimizer.dynamicPartitionPruning.enabled')

In [ ]:
spark.conf.set('spark.sql.optimizer.dynamicPartitionPruning.enabled', False)

no_dpp = (
    users_fact.withColumnRenamed('location', 'country')
    .join(regions_dim.hint('broadcast').filter(col('region').isin(['Europe'])), 'country')
)

no_dpp.explain()

In [ ]:
(
    no_dpp
    .write
    .mode('overwrite')
    .format('noop')
    .save()
)

### Task 4: Enable DPP

Turn DPP back on and re-run the same join. The plan now shows a `dynamicpruningexpression` inside `PartitionFilters` on the fact-side scan — at runtime Spark will broadcast the dimension, collect the distinct `country` values that survive the `region = 'Europe'` filter, and use them to skip non-matching partitions of the fact table.

Hint:
* `spark.sql.optimizer.dynamicPartitionPruning.enabled = true`
* look for a line like `PartitionFilters: [isnotnull(country#...), dynamicpruningexpression(country#... IN dynamicpruning#...)]` in the scan node
* the Stages tab now reports a **smaller** input size for the fact side — only the partitions matching the European countries listed in the dimension are read

In [ ]:
spark.conf.set('spark.sql.optimizer.dynamicPartitionPruning.enabled', True)

with_dpp = (
    users_fact.withColumnRenamed('location', 'country')
    .join(regions_dim.hint('broadcast').filter(col('region').isin(['Europe'])), 'country')
)

with_dpp.explain()

In [ ]:
(
    with_dpp
    .write
    .mode('overwrite')
    .format('noop')
    .save()
)

In [ ]:
spark.stop()